In [4]:
import xml.etree.ElementTree as ET
import pandas as pd
from collections import defaultdict


In [6]:
XML_PATH = "./data/term_dictionaries/drugbank_full_20251115/drugbank_full_database.xml"   # <-- your DrugBank XML file path
DB_NS = "http://www.drugbank.ca"
ns = {"db": DB_NS}



In [10]:

def extract_drug_record(drug_elem):
    """Extract all useful 'name-like' info from a single <drug> element."""
    # --- DrugBank IDs (primary + secondary) ---
    drugbank_ids = [
        e.text.strip()
        for e in drug_elem.findall("db:drugbank-id", ns)
        if e.text
    ]

    primary_id = drugbank_ids[0] if drugbank_ids else None

    # --- Primary name ---
    name_elem = drug_elem.find("db:name", ns)
    primary_name = name_elem.text.strip() if name_elem is not None and name_elem.text else None

    # --- Synonyms ---
    synonyms = [
        e.text.strip()
        for e in drug_elem.findall("db:synonyms/db:synonym", ns)
        if e.text
    ]

    # --- Brand-like names ---

    # International brands
    brand_names = [
        e.text.strip()
        for e in drug_elem.findall("db:international-brands/db:international-brand/db:name", ns)
        if e.text
    ]

    # Product names
    product_names = [
        e.text.strip()
        for e in drug_elem.findall("db:products/db:product/db:name", ns)
        if e.text
    ]

    # --- External identifiers ---
    # For each <external-identifier> get resource + identifier
    ext_ids_map = defaultdict(list)
    for xid in drug_elem.findall("db:external-identifiers/db:external-identifier", ns):
        resource = xid.findtext("db:resource", default="", namespaces=ns) if hasattr(xid, 'findtext') else None
        if resource is None:
            res_elem = xid.find("db:resource", ns)
            resource = res_elem.text.strip() if res_elem is not None and res_elem.text else ""
        id_elem = xid.find("db:identifier", ns)
        identifier = id_elem.text.strip() if id_elem is not None and id_elem.text else ""
        if resource and identifier:
            ext_ids_map[resource].append(identifier)

    # We can store external IDs as a simple "resource:id1|id2;resource2:id3" string
    ext_ids_serialized = ";".join(
        f"{res}:{'|'.join(ids)}" for res, ids in ext_ids_map.items()
    )

    # --- Collect all name-like strings in one set ---
    all_names_set = set()
    for x in (primary_name,) + tuple(synonyms) + \
               tuple(brand_names) + tuple(product_names):
        if x:
            all_names_set.add(x)

    all_names = sorted(all_names_set)

    # Return a flat dict suitable for pandas
    return {
        "primary_drugbank_id": primary_id,
        "all_drugbank_ids": drugbank_ids,
        "primary_name": primary_name,
        "synonyms": synonyms,
        "brand_names": brand_names,
        "product_names": product_names,
        "all_names": all_names,
        "external_identifiers": ext_ids_serialized,
    }


def iter_drugs(xml_path):
    """Stream over <drug> elements one at a time."""
    # iterparse on 'end' events so we see each <drug> complete, then clear it
    context = ET.iterparse(xml_path, events=("end",))
    for event, elem in context:
        # Tags have namespace, so we check full name:
        if elem.tag == f"{{{DB_NS}}}drug":
            yield elem
            # Free memory for this subtree
            elem.clear()


# --- Main: build a DataFrame (without loading all XML in memory) ---

records = []

for drug_elem in iter_drugs(XML_PATH):
    record = extract_drug_record(drug_elem)
    records.append(record)

# Now records is just a list of small dicts, safe for pandas
df = pd.DataFrame(records)


In [14]:
df[df['primary_drugbank_id']=='DB00073'].all_names.iloc[0]

['Blitzima',
 'MabThera',
 'Mabthera',
 'Riabni',
 'Ritemvia',
 'Rituxan',
 'Rituxan Hycela',
 'Rituxan Sc',
 'Rituximab',
 'Rituzena',
 'Rixathon',
 'Riximyo',
 'Ruxience',
 'Truxima',
 'rituximab-abbs',
 'rituximab-arrx',
 'rituximab-pvvr']

In [41]:
df = df.drop_duplicates(subset=["primary_drugbank_id"])

In [43]:
df.shape

(17430, 8)

In [45]:
df.to_csv("./data/term_dictionaries/drugbank_full_20251115/drugbank_full_database_synonyms.csv", index=False)